# Laboratorio 8

## Inciso 1

In [6]:
import sys
import subprocess
import importlib.util
import warnings
from pathlib import Path
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

paquetes = ["pyreadr"]

for paquete in paquetes:
    if importlib.util.find_spec(paquete) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])

In [7]:
import pyreadr

data_dir = Path("data")

rdata_files = sorted(data_dir.glob("*.RData")) + sorted(data_dir.glob("*.rda"))

if not rdata_files:
    rdata_files = sorted(Path(".").glob("*.RData")) + sorted(Path(".").glob("*.rda"))

rdata_path = rdata_files[0]

rdata = pyreadr.read_r(str(rdata_path))

df = next(valor for valor in rdata.values() if isinstance(valor, pd.DataFrame))

df.columns = [str(col).strip() for col in df.columns]

df.shape

(171748, 80)

In [8]:
def limpiar_precio(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce")
    return pd.to_numeric(
        serie.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace(" ", "", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan}),
        errors="coerce"
    )

def preparar_base(datos):
    datos = datos.copy()
    datos.columns = [str(col).strip() for col in datos.columns]

    candidatos_precio = [
        "price", "precio", "Price", "PRICE", "price_usd", "precio_usd",
        "adjusted_price", "listing_price"
    ]

    precio_col = next((col for col in candidatos_precio if col in datos.columns), None)

    if precio_col is None:
        columnas_precio = [
            col for col in datos.columns 
            if "price" in col.lower() or "precio" in col.lower()
        ]
        precio_col = columnas_precio[0]

    datos[precio_col] = limpiar_precio(datos[precio_col])
    datos = datos[datos[precio_col].notna()].copy()

    candidatos_categoria = [
        "categoria_precio", "price_category", "price_cat", "precio_cat",
        "categoria", "categoria_de_precio", "nivel_precio"
    ]

    categoria_col = next((col for col in candidatos_categoria if col in datos.columns), None)

    if categoria_col is None:
        categoria_col = "categoria_precio"
        datos[categoria_col] = pd.qcut(
            datos[precio_col].rank(method="first"),
            q=3,
            labels=["barata", "media", "cara"]
        )
    else:
        datos[categoria_col] = datos[categoria_col].astype(str).str.strip().str.lower()

    return datos, precio_col, categoria_col

df, precio_col, categoria_col = preparar_base(df)

precio_col, categoria_col, df.shape

('price', 'categoria_precio', (76246, 81))

In [9]:
def leer_tabla(path):
    sufijo = path.suffix.lower()
    if sufijo == ".csv":
        return pd.read_csv(path)
    if sufijo in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if sufijo == ".parquet":
        return pd.read_parquet(path)
    if sufijo in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    raise ValueError(path)

extensiones = ["*.csv", "*.xlsx", "*.xls", "*.parquet", "*.pkl", "*.pickle"]

archivos_tabla = []

for extension in extensiones:
    archivos_tabla.extend(sorted(data_dir.glob(extension)))

train_files = [
    path for path in archivos_tabla
    if any(txt in path.stem.lower() for txt in ["train", "entrenamiento", "training"])
]

test_files = [
    path for path in archivos_tabla
    if any(txt in path.stem.lower() for txt in ["test", "prueba", "testing"])
]

if train_files and test_files:
    train_df, _, _ = preparar_base(leer_tabla(train_files[0]))
    test_df, _, _ = preparar_base(leer_tabla(test_files[0]))
    origen_split = "archivos"
else:
    from sklearn.model_selection import train_test_split

    estratificar = df[categoria_col] if df[categoria_col].value_counts().min() > 1 else None

    train_df, test_df = train_test_split(
        df,
        test_size=0.30,
        random_state=2026,
        stratify=estratificar
    )

    origen_split = "semilla"

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

pd.DataFrame({
    "conjunto": ["train", "test"],
    "filas": [train_df.shape[0], test_df.shape[0]],
    "columnas": [train_df.shape[1], test_df.shape[1]],
    "origen_split": [origen_split, origen_split]
})

,conjunto,filas,columnas,origen_split
0,train,53372,81,semilla
1,test,22874,81,semilla


In [10]:
pd.concat([
    train_df[categoria_col].value_counts().rename("train"),
    test_df[categoria_col].value_counts().rename("test")
], axis=1)

,train,test
categoria_precio,,
barata,17791,7625
cara,17791,7624
media,17790,7625


## Inciso 2

In [11]:
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,city,categoria_precio
0,5456.0,https://www.airbnb.com/rooms/5456,2.025092e+13,2025-09-17,city scrape,"Walk to 6th, Rainey St and Convention Ctr",Great central location for walking to Convent...,My neighborhood is ideally located if you want...,https://a0.muscache.com/pictures/14084884/b5a3...,8028,...,4.79,NaN,f,1,1,0,0,3.52,"Austin, Texas",barata
1,6448.0,https://www.airbnb.com/rooms/6448,2.025092e+13,2025-09-17,city scrape,"Secluded Studio @ Zilker - King Bed, Bright & ...","Clean, private space with everything you need ...",The neighborhood is fun and funky (but quiet)!...,https://a0.muscache.com/pictures/airflow/Hosti...,14156,...,4.88,NaN,t,1,1,0,0,1.98,"Austin, Texas",media
2,8502.0,https://www.airbnb.com/rooms/8502,2.025092e+13,2025-09-17,city scrape,Woodland Studio Lodging,Studio rental on lower level of home located i...,,https://a0.muscache.com/pictures/miso/Hosting-...,25298,...,4.63,NaN,f,1,1,0,0,0.28,"Austin, Texas",barata
3,13035.0,https://www.airbnb.com/rooms/13035,2.025092e+13,2025-09-17,city scrape,Historic house in highly walkable East Austin,Comfortable 2 bedroom/2 bathroom home very cen...,East Cesar Chavez is a gentrifying urban area ...,https://a0.muscache.com/pictures/miso/Hosting-...,50793,...,4.95,NaN,f,2,2,0,0,0.11,"Austin, Texas",media
4,22828.0,https://www.airbnb.com/rooms/22828,2.025092e+13,2025-09-16,city scrape,Garage Apartment central SE Austin,"Fully furnished, centrally located, second sto...","wikipedia: East_Riverside-Oltorf,_Austin,_Texas",https://a0.muscache.com/pictures/miso/Hosting-...,56488,...,4.84,NaN,f,1,1,0,0,0.30,"Austin, Texas",barata


In [12]:
pd.DataFrame({
    "filas": [df.shape[0]],
    "columnas": [df.shape[1]],
    "precio": [precio_col],
    "respuesta": [categoria_col]
})

,filas,columnas,precio,respuesta
0,76246,81,price,categoria_precio


In [13]:
pd.DataFrame(df.dtypes.astype(str).value_counts()).reset_index().rename(
    columns={"index": "tipo", "count": "cantidad", 0: "cantidad"}
)

,tipo,cantidad
0,str,36
1,int32,18
2,float64,16
3,object,10
4,category,1


In [14]:
faltantes = pd.DataFrame({
    "columna": df.columns,
    "faltantes": df.isna().sum().values,
    "porcentaje": (df.isna().mean().values * 100).round(2),
    "tipo": df.dtypes.astype(str).values,
    "unicos": df.nunique(dropna=True).values
}).sort_values(["porcentaje", "faltantes"], ascending=False)

faltantes.head(30)

,columna,faltantes,porcentaje,tipo,unicos
49,calendar_updated,76246,100.00,object,0
29,neighbourhood_group_cleansed,38121,50.00,str,9
66,review_scores_accuracy,13417,17.60,float64,158
67,review_scores_cleanliness,13417,17.60,float64,171
68,review_scores_checkin,13417,17.60,float64,143
69,review_scores_communication,13417,17.60,float64,146
70,review_scores_location,13417,17.60,float64,170
71,review_scores_value,13417,17.60,float64,172
65,review_scores_rating,13413,17.59,float64,161
78,reviews_per_month,13413,17.59,float64,1050


In [15]:
df[precio_col].describe().to_frame().T

,count,mean,std,min,25%,50%,75%,max
price,76246.0,750.50922,4250.606945,8.0,120.0,193.0,326.0,50123.0


In [16]:
df.groupby(categoria_col)[precio_col].agg(["count", "min", "median", "mean", "max"]).round(2)

,count,min,median,mean,max
categoria_precio,,,,,
barata,25416,8.0,99.0,95.29,143.0
media,25415,143.0,193.0,196.88,268.0
cara,25415,268.0,437.0,1959.39,50123.0


In [17]:
numeric_cols = df.select_dtypes(include=["number", "bool"]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in [precio_col]]

df[numeric_cols].describe().T.round(2).head(30)

,count,mean,std,min,25%,50%,75%,max
id,76246.0,6.668501e+17,5.673330e+17,6.000000e+00,4.315032e+07,7.717878e+17,1.187202e+18,1.518247e+18
scrape_id,76246.0,2.025092e+13,4.512990e+06,2.025092e+13,2.025092e+13,2.025092e+13,2.025092e+13,2.025093e+13
host_id,76246.0,2.007508e+08,1.981979e+08,2.300000e+01,3.410558e+07,1.134417e+08,3.761767e+08,7.197071e+08
latitude,76246.0,2.962000e+01,8.520000e+00,1.899000e+01,2.127000e+01,3.027000e+01,3.891000e+01,4.239000e+01
longitude,76246.0,-1.211700e+02,3.407000e+01,-1.597200e+02,-1.566800e+02,-1.172000e+02,-8.768000e+01,-7.100000e+01
accommodates,76246.0,4.840000e+00,3.000000e+00,1.000000e+00,2.000000e+00,4.000000e+00,6.000000e+00,1.600000e+01
bathrooms,76232.0,1.620000e+00,9.800000e-01,0.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00,2.000000e+01
minimum_nights,76246.0,9.410000e+00,2.239000e+01,1.000000e+00,1.000000e+00,2.000000e+00,5.000000e+00,7.200000e+02
maximum_nights,76246.0,4.745400e+02,4.191400e+02,1.000000e+00,9.000000e+01,3.650000e+02,1.125000e+03,3.650000e+03
minimum_nights_avg_ntm,76246.0,9.870000e+00,2.182000e+01,1.000000e+00,2.000000e+00,3.000000e+00,5.100000e+00,7.200000e+02


In [18]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != categoria_col]

pd.DataFrame({
    "columna": categorical_cols,
    "unicos": [df[col].nunique(dropna=True) for col in categorical_cols],
    "faltantes": [df[col].isna().sum() for col in categorical_cols]
}).sort_values("unicos", ascending=False).head(30)

,columna,unicos,faltantes
0,listing_url,76246,0
6,picture_url,74476,0
3,name,72180,0
32,amenities,67213,0
4,description,63587,0
43,license,38268,10517
7,host_url,29065,0
17,host_picture_url,28073,0
16,host_thumbnail_url,28073,0
5,neighborhood_overview,26250,0


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def convertir_columnas(datos):
    datos = datos.copy()

    for col in datos.columns:
        if datos[col].dtype == "object":
            valores = datos[col].astype(str).str.lower().str.strip()
            mask = datos[col].notna()
            bool_validos = valores[mask].isin(["t", "f", "true", "false", "yes", "no", "si", "sí", "1", "0"])
            if len(bool_validos) > 0 and bool_validos.mean() >= 0.90:
                datos[col] = valores.map({
                    "t": 1, "true": 1, "yes": 1, "si": 1, "sí": 1, "1": 1,
                    "f": 0, "false": 0, "no": 0, "0": 0
                })

        if datos[col].dtype == "object":
            valores = datos[col].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.replace("%", "", regex=False).str.strip()
            convertida = pd.to_numeric(valores, errors="coerce")
            proporcion = convertida.notna().mean()
            if proporcion >= 0.80:
                datos[col] = convertida

        if datos[col].dtype == "object":
            if any(txt in col.lower() for txt in ["price", "precio", "fee", "deposit", "cost"]):
                datos[col] = limpiar_precio(datos[col])

    return datos

def es_columna_fecha(col):
    nombre = col.lower()
    patrones = [
        "last_scraped",
        "calendar_last_scraped",
        "host_since",
        "first_review",
        "last_review"
    ]
    return (
        nombre in patrones
        or nombre.endswith("_date")
        or nombre.startswith("date_")
        or "_date_" in nombre
        or nombre.endswith("_fecha")
        or nombre.startswith("fecha_")
        or "_fecha_" in nombre
    )

train_df = convertir_columnas(train_df)
test_df = convertir_columnas(test_df)

columnas_fecha = []

for col in train_df.columns:
    if es_columna_fecha(col):
        train_fecha = pd.to_datetime(train_df[col], errors="coerce")
        test_fecha = pd.to_datetime(test_df[col], errors="coerce")
        proporcion = pd.concat([train_fecha, test_fecha]).notna().mean()

        if proporcion >= 0.50:
            referencia = max(train_fecha.max(), test_fecha.max())
            train_df[f"{col}_dias"] = (referencia - train_fecha).dt.days
            test_df[f"{col}_dias"] = (referencia - test_fecha).dt.days
            columnas_fecha.append(col)

columnas_url = [
    col for col in train_df.columns
    if "url" in col.lower() or "picture" in col.lower()
]

columnas_id = [
    col for col in train_df.columns
    if col.lower() == "id" or col.lower().endswith("_id") or col.lower().endswith(" id")
]

columnas_texto = [
    col for col in train_df.select_dtypes(include=["object", "category"]).columns
    if train_df[col].astype(str).str.len().median() > 60
]

columnas_alta_cardinalidad = [
    col for col in train_df.select_dtypes(include=["object", "category"]).columns
    if train_df[col].nunique(dropna=True) > min(200, train_df.shape[0] * 0.35)
]

columnas_sin_info = [
    col for col in train_df.columns
    if train_df[col].isna().all() or train_df[col].nunique(dropna=True) == 0
]

columnas_fuga = [
    col for col in train_df.columns
    if col in [precio_col, categoria_col]
    or "revenue" in col.lower()
    or "income" in col.lower()
]

columnas_eliminar = sorted(set(
    columnas_fuga
    + columnas_fecha
    + columnas_url
    + columnas_id
    + columnas_texto
    + columnas_alta_cardinalidad
    + columnas_sin_info
))

X_train = train_df.drop(columns=columnas_eliminar, errors="ignore")
X_test = test_df.drop(columns=columnas_eliminar, errors="ignore")

y_train = train_df[categoria_col].astype(str)
y_test = test_df[categoria_col].astype(str)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

def crear_encoder():
    opciones = [
        {"handle_unknown": "ignore", "min_frequency": 10, "sparse_output": True},
        {"handle_unknown": "ignore", "min_frequency": 10, "sparse": True},
        {"handle_unknown": "ignore", "sparse_output": True},
        {"handle_unknown": "ignore", "sparse": True}
    ]
    for opcion in opciones:
        try:
            return OneHotEncoder(**opcion)
        except TypeError:
            pass

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", crear_encoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

X_train_svm = preprocessor.fit_transform(X_train)
X_test_svm = preprocessor.transform(X_test)

X_train_svm.shape, X_test_svm.shape


((53372, 322), (22874, 322))

In [20]:
resumen_transformaciones = pd.DataFrame({
    "parte": [
        "respuesta",
        "precio",
        "split",
        "faltantes numericos",
        "faltantes categoricos",
        "variables numericas",
        "variables categoricas",
        "fechas",
        "columnas eliminadas",
        "matriz train",
        "matriz test"
    ],
    "resultado": [
        categoria_col,
        precio_col,
        origen_split,
        "mediana",
        "moda",
        "escaladas",
        "one hot encoding",
        len(columnas_fecha),
        len(columnas_eliminar),
        X_train_svm.shape,
        X_test_svm.shape
    ]
})

resumen_transformaciones

,parte,resultado
0,respuesta,categoria_precio
1,precio,price
2,split,semilla
3,faltantes numericos,mediana
4,faltantes categoricos,moda
5,variables numericas,escaladas
6,variables categoricas,one hot encoding
7,fechas,5
8,columnas eliminadas,27
9,matriz train,"(53372, 322)"


In [21]:
def motivo_eliminacion(col):
    if col in [precio_col, categoria_col]:
        return "respuesta o fuga de información"
    if "revenue" in col.lower() or "income" in col.lower():
        return "posible fuga de información"
    if col in columnas_sin_info:
        return "sin información"
    if col in columnas_fecha:
        return "fecha transformada a dias"
    if col in columnas_url:
        return "url o imagen"
    if col in columnas_id:
        return "identificador"
    if col in columnas_texto:
        return "texto largo"
    return "alta cardinalidad"

pd.DataFrame({
    "columna": columnas_eliminar,
    "motivo": [motivo_eliminacion(col) for col in columnas_eliminar]
})

,columna,motivo
0,amenities,texto largo
1,calendar_last_scraped,fecha transformada a dias
2,calendar_updated,sin información
3,categoria_precio,respuesta o fuga de información
4,description,texto largo
5,estimated_revenue_l365d,posible fuga de información
6,first_review,fecha transformada a dias
7,host_about,texto largo
8,host_id,identificador
9,host_location,alta cardinalidad


In [22]:
salida_dir = data_dir / "procesados_l8"
salida_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(salida_dir / "train_l8.csv", index=False)
test_df.to_csv(salida_dir / "test_l8.csv", index=False)

pd.DataFrame({
    "archivo": [
        str(salida_dir / "train_l8.csv"),
        str(salida_dir / "test_l8.csv")
    ]
})

,archivo
0,data\procesados_l8\train_l8.csv
1,data\procesados_l8\test_l8.csv


## Inciso 3

## Inciso 4

In [38]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['categoria_precio']
)

In [39]:
categoria_col = 'categoria_precio'

X_train = train_df.drop(columns=[categoria_col])
y_train = train_df[categoria_col]

X_test = test_df.drop(columns=[categoria_col])
y_test = test_df[categoria_col]

In [40]:
#QUITAR columnas no numericas o con demasiados valores unicos
cols_drop = [
    'id','listing_url','scrape_id','last_scraped','source','name',
    'description','neighborhood_overview','picture_url','host_url',
    'host_name','host_about','host_thumbnail_url','host_picture_url',
    'amenities','calendar_updated','calendar_last_scraped','license'
]

X_train = X_train.drop(columns=cols_drop, errors='ignore')
X_test  = X_test.drop(columns=cols_drop, errors='ignore')

In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# columnas
num_cols = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','bool']).columns.tolist()

# pipeline numérico
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# pipeline categórico
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# preprocessor final
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
#muestra para entrenamiento
X_train_sample = X_train.sample(5000, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

In [47]:
#modelo SVM
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Lineal
svm_linear = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVC(kernel='linear', C=1))
])

# RBF
svm_rbf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVC(kernel='rbf', C=1, gamma='scale'))
])

# Polinomial
svm_poly = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVC(kernel='poly', degree=3, C=1))
])

In [ ]:
#entrenar modelos
svm_linear.fit(X_train_sample, y_train_sample)
svm_rbf.fit(X_train_sample, y_train_sample)
svm_poly.fit(X_train_sample, y_train_sample)